# Customer Churn Modeling

This notebook tunes and compares three supervised learning models on the provided customer churn training and testing splits:

- Logistic Regression as an interpretable baseline
- Random Forest as a robust non-linear benchmark
- Histogram-based Gradient Boosting as a higher-capacity boosted-tree model

The workflow uses grid search inside the training data, selects a threshold on validation data, refits the best model on the full training split, and evaluates final performance on the untouched test split.

In [4]:
import os
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from customer_churn_analysis.data import load_clean_train_test
from customer_churn_analysis.modeling import (
    FEATURE_COLUMNS,
    build_comparison_table,
    extract_model_signal_table,
    plot_calibration_curves,
    plot_confusion_count_matrices,
    plot_confusion_matrices,
    plot_model_metric_comparison,
    plot_pca_component_projection,
    plot_roc_and_precision_recall,
    plot_train_vs_test_metrics,
    predict_single_customer,
    train_and_compare_models,
    write_model_evaluation_report,
)

plt.style.use("seaborn-v0_8-whitegrid")

In [5]:
datasets = load_clean_train_test()
train_df = datasets["train"]
test_df = datasets["test"]

pd.DataFrame(
    {
        "split": ["train", "test"],
        "rows": [len(train_df), len(test_df)],
        "churn_rate": [train_df["churn"].dropna().mean(), test_df["churn"].dropna().mean()],
    }
).round(4)

,split,rows,churn_rate
0,train,440833,0.5671
1,test,64374,0.4737


## Tune And Compare Models

In [ ]:
evaluations = train_and_compare_models()

Fitting 3 folds for each of 5 candidates, totalling 15 fits
Fitting 3 folds for each of 12 candidates, totalling 36 fits


In [ ]:
comparison_table = build_comparison_table(evaluations)
comparison_table

In [ ]:
pd.DataFrame(
    [
        {
            "model_name": evaluation.model_name,
            "selected_threshold": evaluation.threshold,
            "cv_roc_auc": evaluation.cross_validation_score,
            "best_params": str(evaluation.best_params),
        }
        for evaluation in evaluations
    ]
).round(3)

In [ ]:
plot_model_metric_comparison(comparison_table)
plt.show()

In [ ]:
plot_roc_and_precision_recall(evaluations)
plt.show()

## PCA View

Only the logistic-regression pipeline uses PCA, so the projection below helps show how much of the processed feature variation is captured by the first two components.

In [ ]:
logistic_evaluation = next(evaluation for evaluation in evaluations if evaluation.model_name == "logistic_regression")
plot_pca_component_projection(logistic_evaluation, train_df[FEATURE_COLUMNS], train_df["churn"].astype(int))
plt.show()

In [ ]:
plot_calibration_curves(evaluations)
plt.show()

In [ ]:
plot_train_vs_test_metrics(evaluations)
plt.show()

In [ ]:
plot_confusion_matrices(evaluations)
plt.show()

In [ ]:
plot_confusion_count_matrices(evaluations)
plt.show()

## Reports and Model Signals

In [ ]:
for evaluation in evaluations:
    title = evaluation.model_name.replace("_", " ").title()
    display(Markdown(f"### {title}"))
    print(evaluation.classification_report_text)

In [ ]:
for evaluation in evaluations:
    title = evaluation.model_name.replace("_", " ").title()
    display(Markdown(f"### {title} Signals"))
    display(extract_model_signal_table(evaluation, train_df[FEATURE_COLUMNS], train_df["churn"].astype(int)))

## Example Predictions

The helper below uses the full saved preprocessing-and-model pipeline, which is safer than manually rebuilding encoded feature arrays.

In [ ]:
example_customers = test_df[FEATURE_COLUMNS].head(3).reset_index(drop=True)
best_evaluation = max(evaluations, key=lambda evaluation: evaluation.metrics["roc_auc"])

pd.concat(
    [
        example_customers,
        pd.DataFrame(
            [predict_single_customer(best_evaluation.model, row.to_dict(), threshold=best_evaluation.threshold) for _, row in example_customers.iterrows()]
        ),
    ],
    axis=1,
)

In [ ]:
write_model_evaluation_report(evaluations)
display(Markdown("Saved the comparison report to `reports/report.md`."))